# TUGAS MANDIRI

### Konteks/Skenario
Tim engineering platform e-commerce (skenario yang sama dari Pertemuan 2-3) resmi meminta seluruh proses analisis data yang sebelumnya memakai pandas **dipindahkan ke PySpark**, karena volume data transaksi diperkirakan akan tumbuh sangat besar dalam waktu dekat sehingga pandas (yang memuat semua data ke RAM) tidak lagi memadai. Sebagai data analyst yang baru belajar PySpark, anda ditugaskan membuktikan bahwa seluruh alur analisis dapat direplikasi menggunakan PySpark, **membaca data langsung dari HDFS**.


### Menyiapkan Dataset

In [1]:
# Sel ini membuat dataset baru untuk Tugas Mandiri Pertemuan 4 dan mengunggahnya ke HDFS
import numpy as np
import pandas as pd

np.random.seed(99)
n = 1000
kategori_list = ["Elektronik", "Fashion", "Makanan & Minuman", "Kesehatan & Kecantikan", "Rumah Tangga", "Olahraga"]
kota_list = ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo", "Kebumen"]
metode_bayar_list = ["Transfer Bank", "E-Wallet", "COD", "Kartu Kredit"]
tanggal_range = pd.date_range("2026-09-01", "2026-09-30", freq="D")

data = {
    "order_id": [f"ORD-{3000 + i}" for i in range(n)],
    "tanggal": np.random.choice(tanggal_range, size=n).astype(str),
    "kategori": np.random.choice(kategori_list, size=n),
    "kota": np.random.choice(kota_list, size=n),
    "unit_terjual": np.random.randint(1, 12, size=n),
    "harga_satuan": np.random.choice([20000, 45000, 60000, 90000, 125000, 200000, 350000], size=n),
    "metode_pembayaran": np.random.choice(metode_bayar_list, size=n),
    "rating": np.random.choice([1, 2, 3, 4, 5, np.nan], size=n, p=[0.03, 0.02, 0.10, 0.30, 0.35, 0.20]),
}
df_tugas4 = pd.DataFrame(data)
df_tugas4.to_csv("transaksi_september_2026.csv", index=False)
print(f"Dataset dibuat: {df_tugas4.shape[0]} baris")

# Mengunggah ke HDFS
!hdfs dfs -mkdir -p /user/mahasiswa/tugas4
!hdfs dfs -put -f transaksi_september_2026.csv /user/mahasiswa/tugas4/
print("Berhasil diunggah ke HDFS: /user/mahasiswa/tugas4/transaksi_september_2026.csv")

Dataset dibuat: 1000 baris
Berhasil diunggah ke HDFS: /user/mahasiswa/tugas4/transaksi_september_2026.csv


 ### Memulai SparkSession

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, avg, count

# Menyalakan Spark
spark = SparkSession.builder \
    .appName("Tugas4_PySpark") \
    .getOrCreate()

print("SparkSession berhasil dibuat!")
print("Versi Spark:", spark.version)

26/09/16 19:09:36 WARN Utils: Your hostname, rindani-Latitude-3410 resolves to a loopback address: 127.0.1.1; using 192.168.1.11 instead (on interface wlp0s20f3)
26/09/16 19:09:36 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/16 19:09:37 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


SparkSession berhasil dibuat!
Versi Spark: 3.5.9


## A. Membaca dan Eksplorasi Awal 

In [3]:
# 1. Baca data dari HDFS
hdfs_path = "hdfs://localhost:9000/user/mahasiswa/tugas4/transaksi_september_2026.csv"
df = spark.read.csv(hdfs_path, header=True, inferSchema=True)

# 2. Tampilkan struktur data / tipe kolom
df.printSchema()

# 3. Cek berapa banyak total baris data
print("Total baris data:", df.count())

# 4. Lihat 10 baris pertama
df.show(10)

root
 |-- order_id: string (nullable = true)
 |-- tanggal: timestamp (nullable = true)
 |-- kategori: string (nullable = true)
 |-- kota: string (nullable = true)
 |-- unit_terjual: integer (nullable = true)
 |-- harga_satuan: integer (nullable = true)
 |-- metode_pembayaran: string (nullable = true)
 |-- rating: double (nullable = true)

Total baris data: 1000
+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+
|order_id|            tanggal|            kategori|      kota|unit_terjual|harga_satuan|metode_pembayaran|rating|
+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+
|ORD-3000|2026-09-02 00:00:00|        Rumah Tangga|Yogyakarta|           3|       90000|              COD|   4.0|
|ORD-3001|2026-09-04 00:00:00|   Makanan & Minuman|      Solo|           3|      200000|         E-Wallet|   5.0|
|ORD-3002|2026-09-26 00:00:00|Kesehatan & Kecan...|  Semarang|    

## B. Menangani Data Kosong

In [4]:
# Cek berapa banyak data rating yang kosong
total_kosong = df.filter(col("rating").isNull()).count()
print(f"Jumlah data rating yang kosong: {total_kosong}")

# Mengisi data kosong dengan nilai 0.0 menggunakan fillna
df_clean = df.fillna({"rating": 0.0})

Jumlah data rating yang kosong: 204


*Saya memilih metode fillna(0.0) untuk mengisikan nilai 0.0 pada kolom rating yang kosong. Alasan menggunakan cara ini dibandingkan dropna() adalah agar kita tidak membuang baris data transaksi lainnya. Jika dibuang (dropna), data angka penjualan dan pendapatan di analisis selanjutnya bisa jadi kurang akurat.*

## C. Transformasi Data

In [5]:
from pyspark.sql.functions import col, when

# 1. Menambahkan kolom total_pendapatan = unit_terjual x harga_satuan
df_transformed = df_clean.withColumn("total_pendapatan", col("unit_terjual") * col("harga_satuan"))

# 2. Menambahkan kolom tier_transaksi (Besar jika total_pendapatan > 500000, selain itu Kecil)
df_transformed = df_transformed.withColumn(
    "tier_transaksi",
    when(col("total_pendapatan") > 500000, "Besar").otherwise("Kecil")
)

df_transformed.show(10)

+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+----------------+--------------+
|order_id|            tanggal|            kategori|      kota|unit_terjual|harga_satuan|metode_pembayaran|rating|total_pendapatan|tier_transaksi|
+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+----------------+--------------+
|ORD-3000|2026-09-02 00:00:00|        Rumah Tangga|Yogyakarta|           3|       90000|              COD|   4.0|          270000|         Kecil|
|ORD-3001|2026-09-04 00:00:00|   Makanan & Minuman|      Solo|           3|      200000|         E-Wallet|   5.0|          600000|         Besar|
|ORD-3002|2026-09-26 00:00:00|Kesehatan & Kecan...|  Semarang|           8|       60000|         E-Wallet|   3.0|          480000|         Kecil|
|ORD-3003|2026-09-09 00:00:00|   Makanan & Minuman|  Semarang|           6|      350000|    Transfer Bank|   4.0|         21

## D. Analisis dengan GroupBy

In [6]:
from pyspark.sql.functions import sum as spark_sum, count, avg

# 1. Kategori apa yang memiliki total_pendapatan tertinggi?
df_transformed.groupBy("kategori") \
    .agg(spark_sum("total_pendapatan").alias("total_pendapatan")) \
    .orderBy(col("total_pendapatan").desc()) \
    .show(1)

# 2. Kota mana dengan jumlah transaksi tier "Besar" terbanyak?
df_transformed.filter(col("tier_transaksi") == "Besar") \
    .groupBy("kota") \
    .agg(count("order_id").alias("jumlah_transaksi_besar")) \
    .orderBy(col("jumlah_transaksi_besar").desc()) \
    .show(1)

# 3. Berapa rata-rata rating untuk masing-masing metode_pembayaran?
df_transformed.groupBy("metode_pembayaran") \
    .agg(avg("rating").alias("rata_rata_rating")) \
    .show()

+------------+----------------+
|    kategori|total_pendapatan|
+------------+----------------+
|Rumah Tangga|       138665000|
+------------+----------------+
only showing top 1 row

+----+----------------------+
|kota|jumlah_transaksi_besar|
+----+----------------------+
|Solo|                    92|
+----+----------------------+
only showing top 1 row

+-----------------+------------------+
|metode_pembayaran|  rata_rata_rating|
+-----------------+------------------+
|              COD|3.3745019920318726|
|    Transfer Bank|3.3399209486166006|
|     Kartu Kredit|3.1910569105691056|
|         E-Wallet|             3.292|
+-----------------+------------------+



## E. Menyimpan Hasil ke HDFS 

In [7]:
# Path tujuan penyimpan di HDFS
output_hdfs_path = "hdfs://localhost:9000/user/mahasiswa/tugas4/hasil_transaksi_september"

# Menyimpan DataFrame ke HDFS dalam format CSV
df_transformed.write.mode("overwrite").csv(output_hdfs_path, header=True)

# Verifikasi hasil simpanan dengan membacanya kembali
df_verify = spark.read.csv(output_hdfs_path, header=True, inferSchema=True)
df_verify.show(5)

+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+----------------+--------------+
|order_id|            tanggal|            kategori|      kota|unit_terjual|harga_satuan|metode_pembayaran|rating|total_pendapatan|tier_transaksi|
+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+----------------+--------------+
|ORD-3000|2026-09-02 00:00:00|        Rumah Tangga|Yogyakarta|           3|       90000|              COD|   4.0|          270000|         Kecil|
|ORD-3001|2026-09-04 00:00:00|   Makanan & Minuman|      Solo|           3|      200000|         E-Wallet|   5.0|          600000|         Besar|
|ORD-3002|2026-09-26 00:00:00|Kesehatan & Kecan...|  Semarang|           8|       60000|         E-Wallet|   3.0|          480000|         Kecil|
|ORD-3003|2026-09-09 00:00:00|   Makanan & Minuman|  Semarang|           6|      350000|    Transfer Bank|   4.0|         21

*Spark menyimpan file output dalam beberapa berkas partisi (seperti part-00000...) karena sifat dasarnya yang bekerja secara terdistribusi. Data dibagi-bagi ke beberapa partisi agar proses penulisan data bisa dilakukan secara paralel oleh sistem. Hal ini mempercepat kinerja pemrosesan saat menangani data dalam jumlah sangat besar (Big Data).*

### Menutup SparkSession

In [8]:
# Menutup SparkSession 
spark.stop()
print("SparkSession ditutup.")

SparkSession ditutup.
